# Map Data to Tickers & Combine with Sector Weights

This notebook:
1. Maps Reddit data files to their tickers
2. Combines multiple CSV files into one
3. Merges sentiment data with sector weight information

In [ ]:
# Import required libraries
import pandas as pd
import os

def get_ticker_from_filename(filename):
    """Extract ticker from filename (e.g., 'AMZN_updated.csv' -> 'AMZN')."""
    # Extract basename (e.g., 'AMZN_updated.csv' from 'S&P Updated/AMZN_updated.csv')
    basename = os.path.basename(filename)
    # Split on '_' and take the first part (e.g., 'AMZN' from 'AMZN_updated.csv')
    ticker = basename.split('_')[0]
    return ticker


## Combine CSV Files

Load all CSV files from S&P Updated/ directory and combine into a single file.

In [ ]:
# Specify file paths
sentiment_file = 'sp100_sentiment.csv'
weights_file = 'Weights-05_25_2025.csv'
output_file = 'sp100_sentiment_with_sector.csv'

try:
    # Load sentiment and weights CSVs
    df_sentiment = pd.read_csv(sentiment_file)
    df_weights = pd.read_csv(weights_file)
    
    # Standardize ticker columns (e.g., convert to uppercase)
    df_sentiment['ticker'] = df_sentiment['ticker'].str.upper()
    df_weights['Ticker'] = df_weights['Ticker'].str.upper()
    
    # Merge to add sector column
    df_merged = df_sentiment.merge(
        df_weights[['Ticker', 'Sector']],
        left_on='ticker',
        right_on='Ticker',
        how='left'
    )
    
    # Drop the redundant Ticker column from weights
    df_merged = df_merged.drop(columns=['Ticker'])
    
    # Rename Sector to sector
    df_merged = df_merged.rename(columns={'Sector': 'sector'})
    
    # Reorder columns to place sector after ticker
    columns = ['ticker', 'sector'] + [col for col in df_sentiment.columns if col != 'ticker']
    df_merged = df_merged[columns]
    
    # Identify unmatched tickers
    unmatched_tickers = df_merged[df_merged['sector'].isna()]['tficker'].unique()
    
    # Save the new dataset
    df_merged.to_csv(output_file, index=False)
    print(f"Saved new dataset with sector column to {output_file}")
    
    # Print summary
    print("\nSample of the new dataset (first 5 rows):")
    print(df_merged[['ticker', 'sector'] + df_sentiment.columns[1:].tolist()].head())
    print(f"\nTotal rows in new dataset: {len(df_merged)}")
    print(f"Unmatched tickers (no sector assigned): {len(unmatched_tickers)}")
    if len(unmatched_tickers) > 0:
        print(f"List of unmatched tickers: {unmatched_tickers}")
    
except Exception as e:
    print(f"Error processing files: {e}")

Saved new dataset with sector column to sp100_sentiment_with_sector.csv

Sample of the new dataset (first 5 rows):
  ticker                  sector     type       id parent_post_id  \
0   AAPL  Information Technology     post  18ecxrr            NaN   
1   AAPL  Information Technology  comment  kcmvg9s        18ecxrr   
2   AAPL  Information Technology  comment  kcmxlan        18ecxrr   
3   AAPL  Information Technology  comment  kcn7z6v        18ecxrr   
4   AAPL  Information Technology  comment  kcnfgkx        18ecxrr   

        subreddit                                              title  \
0  wallstreetbets  Apple's head of iphone and Apple watch design ...   
1  wallstreetbets                                                NaN   
2  wallstreetbets                                                NaN   
3  wallstreetbets                                                NaN   
4  wallstreetbets                                                NaN   

                                     